In [1]:
# STEP 1 Analyse the data and pre-processing
import numpy as np  
import pandas as pd 
import os
import matplotlib.pyplot as plt
import seaborn as sns

#Loading the dataset 
DATA_DIR = "C:/Users/anton/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1"
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
print("Fichiers trouvés :", csv_files)

df_list = []

for file in csv_files:
    full_path = os.path.join(DATA_DIR, file)
    print(f"Chargement : {file}")
    df_tmp = pd.read_csv(full_path, low_memory=False)
    df_list.append(df_tmp)

for i, data in enumerate(df_list, start=1):
    rows, cols = data.shape
    print(f'df{i} -> {rows} rows, {cols} columns')
    
data = pd.concat(df_list,axis=0, ignore_index=True)

for df in df_list: del df

#Data Overview 
print(data.shape)


Fichiers trouvés : ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']
Chargement : Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Chargement : Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Chargement : Friday-WorkingHours-Morning.pcap_ISCX.csv
Chargement : Monday-WorkingHours.pcap_ISCX.csv
Chargement : Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Chargement : Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Chargement : Tuesday-WorkingHours.pcap_ISCX.csv
Chargement : Wednesday-workingHours.pcap_ISCX.csv
df1 -> 225745 rows, 79 columns
df2 -> 286467 rows, 79 columns
df3 -> 191033 rows, 79 columns
df4 -> 529918 rows, 79 column

In [3]:
# fixing columns names issues
data.columns = data.columns.str.strip()
data.sample(n=10, random_state=42)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
746827,50545,232,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
946912,53,31226,2,2,68,380,34,34,34.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2216843,80,99951883,9,7,317,11595,317,0,35.222222,105.666667,...,32,999.0,0.0,999,999,99900000.0,0.0,99900000,99900000,DoS Hulk
699389,53,30894,4,2,140,172,35,35,35.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1170268,53,48943,2,2,88,166,44,44,44.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
800686,53,23728,1,1,56,84,56,56,56.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1434488,23,3,2,0,4,0,2,2,2.000000,0.000000,...,24,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1968368,80,141,2,0,0,0,0,0,0.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
934343,443,229,2,0,12,0,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
693547,443,176084,10,8,559,5437,192,0,55.900000,78.196974,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
 # Checking for infinite values
num_columns = data.select_dtypes(include = np.number).columns
has_infinite = np.isinf(data[num_columns]).sum()
print(has_infinite[has_infinite > 0])
# Treating infinite values
data.replace([np.inf, -np.inf], np.nan, inplace=True)

Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64


In [7]:
# The analysis of missing values across the dataset suggests that missing values are not heavily concentrated in any single column 
# and that the dataset can tolerate row-wise removal without significant impact.
data = data.dropna()
data.shape

(2827876, 79)

In [9]:
# Mapping the attacks to the new group
group_mapping = {
    'BENIGN': 'Normal Traffic',
    'DoS Hulk': 'DoS',
    'DDoS': 'DDoS',
    'PortScan': 'Port Scanning',
    'DoS GoldenEye': 'DoS',
    'FTP-Patator': 'Brute Force',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'SSH-Patator': 'Brute Force',
    'Bot': 'Bots',
    'Web Attack � Brute Force': 'Web Attacks',
    'Web Attack � XSS': 'Web Attacks',
    'Infiltration': 'Infiltration',
    'Web Attack � Sql Injection': 'Web Attacks',
    'Heartbleed': 'Miscellaneous'
}

# Map to new group column
data['Attack Type'] = data['Label'].map(group_mapping)

# Checking the new values
data['Attack Type'].value_counts()

# Dropping the old 'Label' column
data.drop(columns='Label', inplace=True)

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time

# --- 1. Data Preparation ---

# Define Features (X) and Target (y)
# 'Attack Type' is the column created in the previous cleaning step
X = data.drop('Attack Type', axis=1)
y = data['Attack Type']

# Encode the target variable (Convert strings to numbers)
# Pipelines handle X, but we usually encode y separately before splitting
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Save the mapping to decode later if needed
label_mapping = dict(zip(le.transform(le.classes_), le.classes_))
print(f"Target Encoding Mapping: {label_mapping}")

# Split the data into training and testing sets
# Stratify=y ensures the class distribution remains consistent in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")


Target Encoding Mapping: {0: 'Bots', 1: 'Brute Force', 2: 'DDoS', 3: 'DoS', 4: 'Infiltration', 5: 'Miscellaneous', 6: 'Normal Traffic', 7: 'Port Scanning', 8: 'Web Attacks'}
Training shape: (1979513, 78)
Testing shape: (848363, 78)


In [13]:
# --- 2. Defining Pipelines ---

# We use a dictionary to store the pipelines. 
# Each pipeline consists of a Scaler (normalization) and the Classifier.
pipelines = {
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        # Ajout de class_weight='balanced'
        ('classifier', DecisionTreeClassifier(class_weight='balanced',max_depth=5, random_state=42))
    ]),
    
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        # Ajout de class_weight='balanced'
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=5,class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        # Ajout de class_weight='balanced' (Crucial pour la régression logistique ici)
        ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000, n_jobs=-1))
    ])
}

# --- 3. Training and Evaluation Loop ---

results = {}

print("\n--- Starting Model Training & Evaluation ---\n")

for name, pipe in pipelines.items():
    print(f"Processing: {name}...")
    start_time = time.time()
    
    # Train the model using the pipeline
    pipe.fit(X_train, y_train)
    
    # Make predictions
    y_pred = pipe.predict(X_test)
    
    # Calculate duration
    duration = time.time() - start_time
    
    # Store metrics
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    # Output results
    print(f"-> {name} trained in {duration:.2f} seconds.")
    print(f"-> Accuracy: {accuracy:.4f}")
    print("-> Classification Report:")
    # target_names applies the original label names to the report
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("-" * 60)
    print("\n")


--- Starting Model Training & Evaluation ---

Processing: Decision Tree...
-> Decision Tree trained in 29.83 seconds.
-> Accuracy: 0.1979
-> Classification Report:
                precision    recall  f1-score   support

          Bots       0.01      0.73      0.01       587
   Brute Force       0.65      1.00      0.78      4150
          DDoS       0.31      1.00      0.47     38407
           DoS       0.10      0.66      0.18     75514
  Infiltration       0.50      1.00      0.67        11
 Miscellaneous       1.00      1.00      1.00         3
Normal Traffic       1.00      0.04      0.08    681396
 Port Scanning       0.36      0.93      0.52     47641
   Web Attacks       0.05      0.90      0.09       654

      accuracy                           0.20    848363
     macro avg       0.44      0.81      0.42    848363
  weighted avg       0.85      0.20      0.14    848363

------------------------------------------------------------


Processing: Random Forest...
-> Random Fo